1. Filtrado del dataset inicial

In [ ]:
import pandas as pd
import requests
import time

def filtrar_compuestos_chembl(input_csv, output_csv):
    # 1. Leer el archivo CSV especificando el separador correcto
    print("Cargando el dataset original...")
    try:
        df = pd.read_csv(input_csv, sep=';')
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo '{input_csv}'. Asegúrate de que esté en la misma carpeta.")
        return
    
    # Limpiar espacios en los nombres de las columnas
    df.columns = df.columns.str.strip()
    
    # Identificar la columna del ID de ChEMBL
    id_column = 'Molecule ChEMBL ID'
    if id_column not in df.columns:
        print(f"Error: No se encontró la columna '{id_column}'. Columnas disponibles: {df.columns.tolist()}")
        return

    # Eliminar duplicados o nulos en los IDs para no hacer peticiones de más
    chembl_ids = df[id_column].dropna().unique().tolist()
    print(f"Se encontraron {len(chembl_ids)} compuestos únicos para analizar.")

    compuestos_filtrados = []
    
    # 2. Configurar la API de ChEMBL
    base_url = "https://www.ebi.ac.uk/chembl/api/data/activity.json"
    
    print("Consultando la API de ChEMBL (esto puede tomar un momento dependiendo del volumen)...")
    
    for idx, chembl_id in enumerate(chembl_ids):
        # Progreso en consola
        if (idx + 1) % 10 == 0 or idx == len(chembl_ids) - 1:
            print(f"Procesando: {idx + 1}/{len(chembl_ids)}...")
            
        # Parámetros de búsqueda para la API
        params = {
            'molecule_chembl_id': chembl_id,
            'limit': 1000, # Traer suficientes registros de actividad por compuesto
            'format': 'json'
        }
        
        try:
            response = requests.get(base_url, params=params, timeout=15)
            if response.status_code == 200:
                data = response.json()
                activities = data.get('activities', [])
                
                es_valido = False
                # 3. Evaluar las actividades del compuesto
                for act in activities:
                    organismo = str(act.get('target_organism', '')).lower()
                    pref_name = str(act.get('target_pref_name', '')).lower()
                    
                    # Comprobar si cumple con las dos condiciones básicas (Plasmodium falciparum + Cysteine Protease)
                    if "plasmodium falciparum" in organismo:
                        if "cysteine protease" in pref_name or "cysteine" in pref_name or "cisteine" in pref_name or "cistein" in pref_name or "cystein" in pref_name  or "falcipain" in pref_name:
                            es_valido = True
                            break # Encontró al menos una coincidencia válida, pasamos al siguiente compuesto
                
                if es_valido:
                    compuestos_filtrados.append(chembl_id)
                    
            else:
                print(f"Advertencia: Error {response.status_code} al consultar el ID {chembl_id}")
                
        except Exception as e:
            print(f"Error de conexión con el ID {chembl_id}: {e}")
        
        # Pequeña pausa para ser amigables con el servidor de ChEMBL
        time.sleep(0.1)

    # 4. Filtrar el DataFrame original y guardar el resultado
    df_resultado = df[df[id_column].isin(compuestos_filtrados)]
    
    df_resultado.to_csv(output_csv, sep=';', index=False)
    print(f"\n¡Proceso completado con éxito!")
    print(f"Compuestos que cumplen el criterio: {len(df_resultado)}")
    print(f"Resultados guardados en: '{output_csv}'")

# --- Ejecución del Script ---
if __name__ == "__main__":
    # Cambia los nombres de los archivos si lo requieres
    archivo_entrada = 'dataset_inicial.csv'
    archivo_salida = 'dataset_filtrado_cysteine_protease.csv'
    
    filtrar_compuestos_chembl(archivo_entrada, archivo_salida)

2. Búsqueda de nuevos compuestos

In [ ]:
import pandas as pd
from chembl_webresource_client.new_client import new_client
from rdkit import Chem

# 1. Definir los clientes de la API
target_api = new_client.target
activity_api = new_client.activity
molecule_api = new_client.molecule

# 2. Buscar dianas de la familia Falcipaina en P. falciparum
targets = target_api.filter(
    organism="Plasmodium falciparum",
    pref_name__icontains="falcipain" or "cysteine protease" 
)

target_ids = [t['target_chembl_id'] for t in targets]
print("Dianas encontradas:", target_ids)

# 3. Obtener actividades asociadas (IC50 <= 10,000 nM / 10 uM)
activities = activity_api.filter(
    target_chembl_id__in=target_ids,
    standard_type="IC50",
    standard_value__lte=10000
)

# 4. Filtrar compuestos no peptídicos
peptide_bond = Chem.MolFromSmarts('[NX3][CX4H]([*])[CX3](=O)[NX3]')

non_peptidic_compounds = []
processed_molecules = set() # Para evitar duplicar compuestos si tienen múltiples ensayos

for act in activities:
    molecule_id = act.get('molecule_chembl_id')
    if not molecule_id or molecule_id in processed_molecules:
        continue
        
    mol_info = molecule_api.get(molecule_id)
    
    # Filtrar solo moléculas pequeñas
    if mol_info.get('molecule_type') != 'Small molecule':
        continue
        
    # Verificar peso molecular < 600 Da
    pref_properties = mol_info.get('molecule_properties') or {}
    mw = float(pref_properties.get('full_mwt', 0) or 0)
    if mw <= 0 or mw > 600:
        continue
        
    # Validar subestructura SMILES para evitar peptidomiméticos
    smiles = (mol_info.get('molecule_structures') or {}).get('canonical_smiles')
    if smiles:
        mol = Chem.MolFromSmiles(smiles)
        if mol and not mol.HasSubstructMatch(peptide_bond):
            processed_molecules.add(molecule_id)
            non_peptidic_compounds.append({
                'chembl_id': molecule_id,
                'target': act.get('target_chembl_id'),
                'ic50_nm': act.get('standard_value'),
                'molecular_weight': mw,
                'smiles': smiles
            })

print(f"Compuestos no peptídicos bioactivos encontrados: {len(non_peptidic_compounds)}")

# =====================================================================
# 5. PASO NUEVO: CONVERTIR A DATAFRAME Y DESCARGAR A CSV
# =====================================================================
# Convertimos la lista de diccionarios en un DataFrame de pandas
df = pd.DataFrame(non_peptidic_compounds)

# Guardamos el DataFrame como archivo CSV
nombre_archivo = "compuestos_falcipaina.csv"

df.to_csv(nombre_archivo, index=False)
print(f"¡Listo! Se ha guardado el archivo en tu equipo como: {nombre_archivo}")

3. Unión de ambos datasets

In [ ]:
import pandas as pd

# 1. Cargar datasets con sus delimitadores correspondientes
df1 = pd.read_csv('compuestos_falcipaina.csv')
df2 = pd.read_csv('dataset_filtrado_cysteine_protease.csv', sep=';')

# 2. Transformaciones previas en Dataset 1
df1_prep = df1.copy()
# Conversión de nM a uM (1 uM = 1000 nM)
df1_prep['activity_uM'] = df1_prep['ic50_nm'] / 1000.0
# Filtrar columnas requeridas
df1_prep = df1_prep[['chembl_id', 'smiles', 'activity_uM', 'molecular_weight']]

# 3. Transformaciones previas en Dataset 2
df2_prep = df2.rename(columns={
    'Molecule ChEMBL ID': 'chembl_id',
    'smiles_std': 'smiles'
})[['chembl_id', 'smiles', 'activity_uM', 'Label']]

# 4. Fusión externa (outer) sobre la columna 'chembl_id'
merged = pd.merge(df1_prep, df2_prep, on='chembl_id', how='outer', suffixes=('_df1', '_df2'))

# 5. Consolidar columnas que coinciden en ambos datasets
merged['smiles'] = merged['smiles_df1'].fillna(merged['smiles_df2'])
merged['activity_uM'] = merged['activity_uM_df2'].fillna(merged['activity_uM_df1'])

# 6. Seleccionar y ordenar las 5 columnas finales requeridas
columnas_finales = ['chembl_id', 'smiles', 'activity_uM', 'Label', 'molecular_weight']
df_final = merged[columnas_finales]

# 7. Guardar el archivo en formato CSV
df_final.to_csv('dataset_unificado.csv', index=False)

print(f"Dataset generado exitosamente con {len(df_final)} filas totales.")

Dataset generado exitosamente con 127 filas totales.


4. Etiquetado en función de la actividad de los compuestos

In [ ]:
import pandas as pd

# 1. Cargar el dataset unificado que habíamos generado
df = pd.read_csv('dataset_unificado.csv')

# 2. Recalcular la columna Label según el umbral de 1 uM
df['Label'] = (df['activity_uM'] < 1.0).astype(int)

# 3. Guardar el dataset final
df.to_csv('dataset_etiquetado.csv', index=False)

# Mostrar distribución de las clases
print(df['Label'].value_counts())

Label
0    99
1    28
Name: count, dtype: int64
